# Results for Final Evaluation on Test Split

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, PercentFormatter

COLORS = {
    "RANDOM_SAMPLING": "#333333",
    "COARSE_UNCERTAINTY_MEASURE": "#E69F00",
    "COARSE_ENTROPY_MEASURE": "#D55E00",
    "GROUPED_FINE_ENTROPY": "#009E73",
    "FINE_UNCERTAINTY_MEASURE": "#56B4E9",
    "FINE_ENTROPY_MEASURE": "#0072B2",
    "WEIGHTED_UNCERTAINTY_MEASURE": "#CC79A7",
    "WEIGHTED_ENTROPY_MEASURE": "#7B3294",
}
LABELS = {
    "RANDOM_SAMPLING": "Random Sampling (Baseline)",
    "COARSE_UNCERTAINTY_MEASURE": "Coarse UM",
    "COARSE_ENTROPY_MEASURE": "Coarse EM",
    "FINE_UNCERTAINTY_MEASURE": "Fine UM",
    "FINE_ENTROPY_MEASURE": "Fine EM",
    "GROUPED_FINE_ENTROPY": "Grouped Fine EM",
    "WEIGHTED_UNCERTAINTY_MEASURE": "Weighted UM",
    "WEIGHTED_ENTROPY_MEASURE": "Weighted EM",
}
GRANULARITIES = ["G_high", "G_mid", "G_low"]
TITLES = [r"$G_{\mathrm{high}}$", r"$G_{\mathrm{mid}}$", r"$G_{\mathrm{low}}$"]
METRICS = {"test_f1_event": "Mean F1 Event",
           "test_f1_element": "Mean F1 Element", "test_edit": "Edit Score"}
CLASS_AGNOSTIC = "test_f1_class_agnostic"
EXPECTED_ROUNDS = set(range(10))
# Sample SD across seeds; undefined for a single seed (no band is drawn).
SD_DDOF = 1
plt.rcParams.update({
    "figure.dpi": 150, "font.size": 11,
    "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18, "axes.axisbelow": True,
    "lines.linewidth": 1.4, "pdf.fonttype": 42, "ps.fonttype": 42,
})
relative = Path("experiments/4_active_learning_runs")
EXPERIMENT_DIR = next((base / relative for base in [Path.cwd(), *Path.cwd().parents]
                       if (base / relative).is_dir()), None)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the alpes repository.")
RUNS_DIR = EXPERIMENT_DIR / "runs"
if not RUNS_DIR.is_dir():
    RUNS_DIR = EXPERIMENT_DIR


def read_json(path):
    with path.open() as stream:
        return json.load(stream)


def load_results(runs_dir):
    records, inventory = [], []
    for config_path in sorted(runs_dir.glob("*/config.json")):
        config = read_json(config_path)
        run_dir = config_path.parent
        strategy = config["query_strategy"]
        if strategy not in COLORS:
            print(f"Skipping unknown strategy: {run_dir.name}")
            continue
        total = config["total_training_frames"]
        if total <= 0:
            raise ValueError(f"Invalid training frame count: {run_dir}")
        history_path = run_dir / "history.json"
        history = {int(row["round"]): row for row in read_json(history_path)} if history_path.exists() else {}
        loaded = set()
        for round_number in sorted(EXPECTED_ROUNDS):
            round_dir = run_dir / f"round_{round_number:03d}"
            result_path = round_dir / "test_results.json"
            if not result_path.exists():
                continue
            try:
                results = read_json(result_path)
                # Frame counts preserve the true annotation budget without rounding.
                frames = history.get(round_number, {}).get("labeled_pool_frames")
                if frames is None:
                    frames = sum(item["num_frames"] for item in read_json(round_dir / "labeled_train.json"))
                if not 0 < frames <= total:
                    raise ValueError("Labeled frame count is outside the training pool")
                row = {"run": run_dir.name, "seed": config["seed"], "strategy": strategy,
                       "round": round_number, "budget_pct": 100.0 * frames / total}
                for granularity in GRANULARITIES:
                    for metric in METRICS:
                        value = float(results[granularity][metric])
                        upper = 100 if metric == "test_edit" else 1
                        if not np.isfinite(value) or not 0 <= value <= upper:
                            raise ValueError(f"Invalid {granularity}/{metric}: {value}")
                        row[f"{granularity}/{metric}"] = value
                value = float(results.get(CLASS_AGNOSTIC, results["G_high"].get(CLASS_AGNOSTIC)))
                if not np.isfinite(value) or not 0 <= value <= 1:
                    raise ValueError("Invalid class-agnostic F1")
                row[CLASS_AGNOSTIC] = value
            except (OSError, ValueError, KeyError, TypeError) as exc:
                print(f"Skipping {run_dir.name}/{round_dir.name}: {exc}")
                continue
            records.append(row)
            loaded.add(round_number)
        missing = sorted(EXPECTED_ROUNDS - loaded)
        inventory.append({"run": run_dir.name, "strategy": strategy, "seed": config["seed"],
                          "loaded_rounds": len(loaded), "complete": not missing,
                          "missing_rounds": missing})
        if missing:
            print(f"{run_dir.name}: missing/unusable AL rounds {missing}")
    rounds = pd.DataFrame(records)
    inventory = pd.DataFrame(inventory)
    if rounds.empty:
        raise ValueError(f"No usable test results found in {runs_dir}")
    if inventory.duplicated(["strategy", "seed"]).any():
        raise ValueError("Multiple runs for a strategy/seed; select one configuration before aggregating.")
    for run, curve in rounds.groupby("run"):
        if not np.all(np.diff(curve.sort_values("round")["budget_pct"]) > 0):
            raise ValueError(f"Annotation budgets must increase strictly: {run}")
    print(f"Loaded {rounds['run'].nunique()} / {len(inventory)} runs, {len(rounds)} rounds; "
          f"{inventory['complete'].sum()} complete runs. Round numbers are zero-based (0–9).")
    return rounds, inventory


rounds, inventory = load_results(RUNS_DIR)
coverage = rounds.groupby(["strategy", "round"])["seed"].nunique().unstack(fill_value=0)
display(coverage.reindex(COLORS, fill_value=0).rename(index=LABELS).rename_axis("Seeds by AL round"))


def summarize(data, score):
    # Align seeds by acquisition round, using the unrounded mean actual budget.
    return data.groupby("round").agg(
        budget=("budget_pct", "mean"), budget_sd=("budget_pct", "std"),
        score=(score, "mean"), score_sd=(score, "std"), seeds=("seed", "nunique"),
    ).reindex(range(10))  # Missing rounds remain gaps, never synthetic observations.


def plot_curve(ax, data, score, relative=False):
    for strategy, color in COLORS.items():
        selected = data.loc[data["strategy"] == strategy]
        if selected.empty:
            continue
        summary = summarize(selected, score)
        budget = summary["budget"].to_numpy()
        mean = summary["score"].to_numpy()
        sd = summary["score_sd"].to_numpy()
        style = "--" if strategy == "RANDOM_SAMPLING" else "-"
        ax.plot(budget, mean, style, color=color, marker="o", markersize=3, clip_on=False)
        ax.fill_between(budget, mean - sd, mean + sd, color=color, alpha=0.12)
    ax.set(xlim=(0, 100), xticks=np.arange(0, 101, 10))
    ax.xaxis.set_major_formatter(PercentFormatter(100))
    if relative:
        ax.yaxis.set_major_formatter(PercentFormatter(100))
        ax.axhline(100, color="0.6", linestyle=":", linewidth=1)
    if data.empty:
        ax.text(0.5, 0.5, "No eligible runs", ha="center", transform=ax.transAxes)


def finish_figure(fig, data, filename):
    present = set(data["strategy"])
    handles = [Line2D([0], [0], color=color, linestyle="--" if strategy == "RANDOM_SAMPLING" else "-",
                      label=LABELS[strategy]) for strategy, color in COLORS.items() if strategy in present]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, fontsize=10)
    # Leave room for endpoint ticks, axis labels, and the shared legend.
    # fig.tight_layout(pad=2.0, w_pad=2.5, h_pad=2.5, rect=(0, 0.11, 1, 0.96))
    fig.tight_layout(
        pad=1.0,
        w_pad=2.5,
        h_pad=1.2,
        rect=(0, 0.05, 1, 0.985),
    )
    output_dir = EXPERIMENT_DIR / "test_results_figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    fig.savefig(output_path, format="pdf", bbox_inches="tight")
    print(f"Saved {output_path}")
    plt.show()

def full_budget_value(data, score):
    # Each seed contributes once, regardless of how many strategies reached 100%.
    full = data.loc[(data["round"] == 9) & np.isclose(data["budget_pct"], 100)]
    if full.empty:
        raise ValueError(f"No full-budget result available for {score}")
    for seed, values in full.groupby("seed")[score]:
        if not np.allclose(values, values.iloc[0], rtol=0, atol=1e-12):
            raise ValueError(f"Full-budget scores differ across strategies for seed {seed}: {score}")
    return full.groupby("seed")[score].first().mean()


def add_full_budget_references(ax, data, score, fractions=(1.0, 0.95, 0.90),
                               right_fractions=()):
    reference = full_budget_value(data, score)
    for fraction in fractions:
        value = fraction * reference
        ax.axhline(value, color="0.4", linestyle=":", linewidth=1, zorder=2.5)
        on_right = fraction in right_fractions
        ax.annotate(
            f"{fraction:.0%} of full-budget score", xy=(0.98 if on_right else 0.02, value),
            xycoords=ax.get_yaxis_transform(), xytext=(0, 3),
            textcoords="offset points", ha="right" if on_right else "left", va="bottom", fontsize=8,
            color="0.3", bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1),
        )



## Learning Curves

One 3×1 figure per metric (Mean F1 Event, Mean F1 Element, and Edit Score), with G_high at the top, G_mid in the middle, and G_low at the bottom. Points use the unrounded mean actual frame budget across available seeds at each round; shaded bands show ±1 sample standard deviation in performance. Missing rounds remain gaps. Seed counts are shown in the setup coverage table.

Within each metric figure, panels use explicit y-axis limits and heights proportional to their y-axis spans, preserving equal score units per vertical inch across granularities. Every panel shows annotation-budget tick values at 10% intervals. F1 grid lines are spaced by 0.01, with ticks and labels only at multiples of 0.05; Edit Score grid lines, ticks, and labels are spaced by 1. All figures are saved as separate PDFs in `test_results_figures/` beside this notebook.


In [ ]:
from matplotlib.ticker import FormatStrFormatter

# Figure sizes in inches (width, height), independently configurable per metric.
LEARNING_CURVE_FIGSIZES = {
    "test_f1_event": (10, 17),
    "test_f1_element": (10, 17),
    "test_edit": (10, 17),
}

# Heights follow the requested spans so each metric has a consistent physical scale.
LEARNING_CURVE_LIMITS = {
    "test_f1_event": [(0.23, 0.44), (0.28, 0.54), (0.52, 0.70)],
    "test_f1_element": [(0.60, 0.77), (0.62, 0.79), (0.71, 0.81)],
    "test_edit": [(60, 75), (71, 83), (77, 89)],
}
# Move labels that intersect early-round curves or their uncertainty bands.
RIGHT_REFERENCE_LABELS = {
    "G_low/test_f1_event": (0.90,),
    "G_high/test_f1_element": (0.90,),
    "G_mid/test_f1_element": (0.90,),
    "G_low/test_f1_element": (0.95, 0.90),
    "G_high/test_edit": (0.90,),
    "G_mid/test_edit": (0.90,),
    "G_low/test_edit": (0.95, 0.90),
}
for metric, label in METRICS.items():
    limits = LEARNING_CURVE_LIMITS[metric]
    spans = [upper - lower for lower, upper in limits]
    fig, axes = plt.subplots(
        3, 1, figsize=LEARNING_CURVE_FIGSIZES[metric], sharex=True, sharey=False,
        gridspec_kw={"height_ratios": spans},
    )
    for ax, granularity, title, bounds in zip(axes, GRANULARITIES, TITLES, limits):
        plot_curve(ax, rounds, f"{granularity}/{metric}")
        ax.set(title=title, ylabel=label, ylim=bounds)
        add_full_budget_references(
            ax, rounds, f"{granularity}/{metric}",
            right_fractions=RIGHT_REFERENCE_LABELS.get(f"{granularity}/{metric}", ()),
        )
        ax.tick_params(axis="x", which="major", labelbottom=True)
        if metric == "test_edit":
            ax.yaxis.set_major_locator(MultipleLocator(1))
            ax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))
        else:
            ax.yaxis.set_major_locator(MultipleLocator(0.02))
            ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
            ax.yaxis.set_minor_locator(MultipleLocator(0.01))
            # Minor locations supply grid lines without visible tick marks or labels.
            ax.tick_params(axis="y", which="minor", left=False, right=False,
                           labelleft=False, labelright=False)
            ax.grid(True, axis="y", which="minor", alpha=0.18)
        ax.grid(True, which="major", axis="both", alpha=0.18)
    axes[-1].set_xlabel("Annotation budget (% of training frames)")
    fig.suptitle(
        f"{label} vs. Annotation Budget\n"
        f"Mean ± 1 SD Across {rounds['seed'].nunique()} Seeds"
    )
    finish_figure(fig, rounds, f"learning_curves_{metric}.pdf")


## Area Under the Learning Curve (AULC)

Compute trapezoidal AULC separately for each complete seed, using its exact budgets and normalizing by its actual first-to-last interval (approximately 10–100%). Report mean ±1 sample SD on a 0–100 scale, with relative percentage change against the random-sampling mean. A single seed has undefined SD (`N/A`).


In [ ]:
def calculate_aulc(data, run_inventory, scores):
    complete_runs = run_inventory.loc[run_inventory["complete"], "run"]
    records = []
    for run, curve in data.loc[data["run"].isin(complete_runs)].groupby("run"):
        curve = curve.sort_values("budget_pct")
        budgets = curve["budget_pct"].to_numpy()
        if set(curve["round"]) != EXPECTED_ROUNDS or not np.isclose(budgets[-1], 100):
            raise ValueError(f"Complete AULC run must contain rounds 0–9 and reach 100%: {run}")
        record = {"run": run, "strategy": curve["strategy"].iloc[0], "seed": curve["seed"].iloc[0]}
        for score in scores:
            values = curve[score].to_numpy()
            # Edit score is already 0–100; F1 must be scaled by 100.
            scale = 1 if score.endswith("/test_edit") else 100
            area = np.sum(0.5 * (values[:-1] + values[1:]) * np.diff(budgets))
            record[score] = scale * area / (budgets[-1] - budgets[0])
        records.append(record)
    return pd.DataFrame(records, columns=["run", "strategy", "seed", *scores])


def aulc_table(aulcs, columns):
    baseline = aulcs.loc[aulcs["strategy"] == "RANDOM_SAMPLING"]
    rows = []
    for strategy, label in LABELS.items():
        selected = aulcs.loc[aulcs["strategy"] == strategy]
        row = {"Query Strategy": label}
        for score, title in columns.items():
            values = selected[score].astype(float)
            reference = baseline[score].astype(float).mean()
            if values.empty:
                row[title] = "—"
                continue
            mean, sd = values.mean(), values.std(ddof=SD_DDOF)
            sd_text = f"{sd:.2f}" if np.isfinite(sd) else "N/A"
            gain = f"{100 * (mean / reference - 1):+.2f}%" if reference > 0 else "N/A"
            row[title] = f"{mean:.2f} ± {sd_text} ({gain})"
        row["No. of Seeds"] = selected["seed"].nunique()
        rows.append(row)
    return pd.DataFrame(rows).set_index("Query Strategy")


score_columns = [f"{g}/{m}" for g in GRANULARITIES for m in METRICS] + [CLASS_AGNOSTIC]
aulcs = calculate_aulc(rounds, inventory, score_columns)
print("AULC: complete runs only; normalized over each run's actual first-to-last budget interval.")
print("Cells: mean ± sample SD (relative increase over random-sampling mean); N/A SD means one seed.")
aulc_tables = {}
for metric, label in METRICS.items():
    display(Markdown(f"### {label}"))
    table = aulc_table(aulcs, {f"{g}/{metric}": title for g, title in zip(GRANULARITIES, TITLES)})
    aulc_tables[metric] = table
    display(table)


## Class-Agnostic PES

In [ ]:
from matplotlib.ticker import FormatStrFormatter

fig, ax = plt.subplots(figsize=(10, 9))
plot_curve(ax, rounds, CLASS_AGNOSTIC)
ax.set(ylabel="Mean F1 Class-Agnostic",
       xlabel="Annotation budget (% of training frames)", ylim=(0.80, 0.88))
add_full_budget_references(ax, rounds, CLASS_AGNOSTIC, fractions=(1.0, 0.95),
                           right_fractions=(0.95,))
ax.yaxis.set_major_locator(MultipleLocator(0.02))
ax.yaxis.set_minor_locator(MultipleLocator(0.01))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.tick_params(axis="x", which="major", labelbottom=True)
ax.tick_params(axis="y", which="minor", left=False, right=False,
               labelleft=False, labelright=False)
ax.grid(True, axis="y", which="minor", alpha=0.18)
ax.grid(True, axis="both", which="major", alpha=0.18)
fig.suptitle(
    "Class-Agnostic PES vs. Annotation Budget\n"
    f"Mean ± 1 SD Across {rounds['seed'].nunique()} Seeds"
)
present = set(rounds["strategy"])
handles = [
    Line2D([0], [0], color=color,
           linestyle="--" if strategy == "RANDOM_SAMPLING" else "-",
           label=LABELS[strategy])
    for strategy, color in COLORS.items() if strategy in present
]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, fontsize=10)
# Reserve the bottom 16% for the four-row legend and separation from the x-axis label.
fig.tight_layout(pad=1.0, w_pad=2.5, h_pad=1.2, rect=(0, 0.1, 1, 0.985))
output_dir = EXPERIMENT_DIR / "test_results_figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "learning_curves_class_agnostic_pes.pdf"
fig.savefig(output_path, format="pdf", bbox_inches="tight")
print(f"Saved {output_path}")
plt.show()


In [ ]:
class_agnostic_aulc = aulc_table(aulcs, {CLASS_AGNOSTIC: "AULC"})
display(class_agnostic_aulc)
